# Week 5 — Evaluation and Optimization

**Goal:** measure the Week 2–4 RAG pipeline — retrieval quality, latency, and prompt behaviour — without touching any core logic. This notebook only adds measurement cells on top of what was already built.

## Pipeline constraints (weeks 2–4, unchanged)

The operational pipeline uses `HashingVectorizer` (768-dim, deterministic, no download) instead of SentenceTransformer because loading MiniLM/MPNet caused kernel crashes during Week 2 demos. The hashing pipeline is **frozen** — this notebook measures it, does not refactor it.

Two evaluation tracks are reported:

| Track | What | Why |
|-------|------|-----|
| Hashing baseline | HashingVectorizer + FAISS, full corpus | Production stability |
| Chunking variant | Same embedder, overlap 50 → 75 | Isolate the effect of chunk boundary cuts |

In [ ]:
import json
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Suppress FAISS threading warnings — not needed for single-query notebook use
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
faiss.omp_set_num_threads(1)

In [ ]:
DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
ART_DIR  = Path('../artifacts') if Path('../artifacts').exists() else Path('artifacts')
ART_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline config — same values used in weeks 2–4
EMBEDDING_MODEL_ID = "hashing-768-stable"
CHUNK_SIZE    = 300
CHUNK_OVERLAP = 50
TOP_K         = 3
N_ITER_LAT    = 20  # iterations per query for latency measurement

EVAL_PATH = Path("../eval/eval.jsonl")
if not EVAL_PATH.exists():
    EVAL_PATH = Path("eval/eval.jsonl")


def save_df(df: pd.DataFrame, filename: str) -> Path:
    out = ART_DIR / filename
    df.to_csv(out, index=False)
    print(f"Saved: {out}")
    return out


print(f"CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}, TOP_K={TOP_K}, embedder={EMBEDDING_MODEL_ID}")

## 1) Pipeline infrastructure

Same HashingVectorizer + FAISS stack from weeks 2–4 — defined here so this notebook is self-contained. `DocumentChunk` stores a chunk with its source metadata. `recover()` embeds a query and returns ranked chunks. `evaluate()` scores retrieval against gold labels from `eval.jsonl`.

In [ ]:
@dataclass
class DocumentChunk:
    chunk_id: int
    text:     str
    source:   str
    topic:    str
    page:     int


class StableEmbedder:
    """HashingVectorizer wrapper — deterministic 768-dim embeddings, no model download."""
    def __init__(self, n_features: int = 768):
        self.vectorizer = HashingVectorizer(
            n_features=n_features, alternate_sign=False, lowercase=True
        )

    def encode(self, texts) -> np.ndarray:
        if isinstance(texts, str):
            texts = [texts]
        return np.ascontiguousarray(
            self.vectorizer.transform(texts).toarray(), dtype=np.float32
        )


def load_and_chunk(data_dir: Path, chunk_size: int, chunk_overlap: int) -> list:
    """Walk data_dir/<topic>/*.pdf and return a flat list of DocumentChunk objects."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    result, cid = [], 0
    for topic_dir in sorted(Path(data_dir).iterdir()):
        if not topic_dir.is_dir():
            continue
        for pdf in topic_dir.glob("*.pdf"):
            try:
                for page in PyPDFLoader(str(pdf)).load():
                    for part in splitter.split_text(page.page_content):
                        if part.strip():
                            result.append(DocumentChunk(
                                chunk_id=cid, text=part,
                                source=pdf.name, topic=topic_dir.name,
                                page=int(page.metadata.get("page", 0)),
                            ))
                            cid += 1
            except Exception as e:
                print(f"Skipping {pdf.name}: {e}")
    return result


def build_index(chunks: list, emb: StableEmbedder) -> faiss.IndexFlatIP:
    """Embed all chunks and load them into a FAISS IndexFlatIP."""
    vecs = emb.encode([c.text for c in chunks])
    idx  = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return idx


def recover(query: str, top_k: int = TOP_K, topic: str | None = None,
            _chunks=None, _index=None, _embedder=None) -> list:
    """Return top-k (DocumentChunk, ref_dict, score) tuples for a query.
    Pass _chunks/_index/_embedder to query an alternative index without swapping globals."""
    c   = _chunks   if _chunks   is not None else chunks
    idx = _index    if _index    is not None else faiss_index
    emb = _embedder if _embedder is not None else embedder

    qv      = emb.encode([query])
    fetch_k = min((top_k * 5) if topic else top_k, idx.ntotal)
    scores, indices = idx.search(qv, fetch_k)

    out = []
    for s, i in zip(scores[0], indices[0]):
        ch = c[int(i)]
        if topic is not None and ch.topic.lower() != topic.lower():
            continue
        out.append((ch, {"source": ch.source, "topic": ch.topic,
                         "page": ch.page, "chunk_id": ch.chunk_id}, float(s)))
        if len(out) >= top_k:
            break
    return out


def _hit_correct(chunk, ref, gold_chunk_ids, gold_sources, gold_keywords) -> bool:
    """Return True if the chunk matches any gold criterion."""
    if gold_chunk_ids and chunk.chunk_id in gold_chunk_ids:
        return True
    if gold_sources and any(gs.lower() in ref["source"].lower() for gs in gold_sources):
        return True
    if gold_keywords and any(kw.lower() in chunk.text.lower() for kw in gold_keywords):
        return True
    return False


def evaluate(eval_path: str, top_k: int = 5, use_topic: bool = True,
             _chunks=None, _index=None, _embedder=None) -> dict:
    """Score retrieval over eval.jsonl. Returns hit@1/3/5 and MRR@5.
    Queries without gold labels (out-of-scope) are counted in n but not scored."""
    hit1, hit3, hit5, mrr5, failures = [], [], [], [], []
    n_eval = n_skip = 0

    for line in Path(eval_path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            n_skip += 1
            continue

        query = str(row.get("query") or row.get("question", "")).strip()
        if not query:
            n_skip += 1
            continue

        topic    = row.get("topic") if use_topic else None
        gids     = row.get("gold_chunk_ids") or []
        gsrc     = row.get("gold_sources")   or []
        gkw      = row.get("gold_keywords")  or []
        has_gold = bool(gids or gsrc or gkw)

        hits    = recover(query, top_k=top_k, topic=topic,
                          _chunks=_chunks, _index=_index, _embedder=_embedder)
        correct = [r for r, (c, ref, _) in enumerate(hits[:5], 1)
                   if _hit_correct(c, ref, gids, gsrc, gkw)]

        if has_gold:
            best = correct[0] if correct else None
            hit1.append(1.0 if best == 1 else 0.0)
            hit3.append(1.0 if best is not None and best <= 3 else 0.0)
            hit5.append(1.0 if best is not None else 0.0)
            mrr5.append(1.0 / best if best else 0.0)
            if not correct:
                failures.append({"id": row.get("id"), "query": query,
                                 "top_sources":   [ref["source"]    for _, ref, _ in hits],
                                 "top_chunk_ids": [ch.chunk_id      for ch, _, _ in hits]})
        n_eval += 1

    def _m(lst): return round(sum(lst) / len(lst), 4) if lst else 0.0
    return {"n": n_eval, "skipped": n_skip,
            "hit@1_mean": _m(hit1), "hit@3_mean": _m(hit3),
            "hit@5_mean": _m(hit5), "mrr@5_mean": _m(mrr5),
            "failures": failures}


print("Infrastructure ready: DocumentChunk, StableEmbedder, load_and_chunk, build_index, recover, evaluate")

In [ ]:
embedder    = StableEmbedder()
chunks      = load_and_chunk(DATA_DIR, CHUNK_SIZE, CHUNK_OVERLAP)
faiss_index = build_index(chunks, embedder)
print(f"Indexed {len(chunks)} chunks  |  FAISS ntotal={faiss_index.ntotal}  |  dim={faiss_index.d}")

## 2) Retrieval evaluation — baseline

`evaluate()` runs every query from `eval.jsonl` through retrieval only and scores each result against gold labels (`gold_sources` or `gold_keywords`). It returns:

- **hit@k** — fraction of queries where a correct chunk appeared in the top-k results
- **MRR@5** — mean reciprocal rank; rewards finding the correct chunk earlier in the list

A hit@3 ≥ 0.6 is the minimum acceptable bar for this demo system.

In [ ]:
report = evaluate(str(EVAL_PATH), top_k=5, use_topic=True)

summary = pd.DataFrame([
    {"metric": k, "value": report[k]}
    for k in ("n", "skipped", "hit@1_mean", "hit@3_mean", "hit@5_mean", "mrr@5_mean")
])
display(summary)

n_fail = len(report["failures"])
print(f"Failures: {n_fail}. Hit@3 acceptable: {'yes' if report['hit@3_mean'] >= 0.6 else 'needs improvement'}.")
if n_fail:
    print("First 5 failures:", report["failures"][:5])

## 3) Topic filtering comparison

`evaluate()` accepts `use_topic=True/False`. When `True`, retrieval is restricted to the PDF folder matching the query's topic (e.g. `rag/`, `git/`, `gcp/`). This reduces cross-topic noise but fails if the topic label is missing or wrong. We compare both modes to decide whether filtering helps.

In [ ]:
report_no_topic   = evaluate(str(EVAL_PATH), top_k=5, use_topic=False)
report_with_topic = report  # reuse result from the baseline cell above

print(pd.DataFrame([
    {"scenario": "no topic",   **{k: report_no_topic[k]   for k in ("hit@1_mean", "hit@3_mean", "hit@5_mean", "mrr@5_mean")}},
    {"scenario": "with topic", **{k: report_with_topic[k] for k in ("hit@1_mean", "hit@3_mean", "hit@5_mean", "mrr@5_mean")}},
]).to_string(index=False))

## 4) Chunking comparison — overlap 50 vs 75

Larger overlap means adjacent chunks share more text, which reduces the chance of a relevant sentence being cut across a boundary. We rebuild the index with `chunk_overlap=75` (all other settings identical) and compare retrieval metrics.

In [ ]:
report_default = report_with_topic  # overlap=50, already evaluated above

# Rebuild index with larger overlap — same embedder, only chunk boundaries change
CHUNK_OVERLAP_ALT = 75
chunks_alt      = load_and_chunk(DATA_DIR, CHUNK_SIZE, CHUNK_OVERLAP_ALT)
faiss_index_alt = build_index(chunks_alt, embedder)
report_overlap75 = evaluate(str(EVAL_PATH), top_k=5, use_topic=True,
                             _chunks=chunks_alt, _index=faiss_index_alt, _embedder=embedder)

METRIC_KEYS = ("hit@1_mean", "hit@3_mean", "hit@5_mean", "mrr@5_mean")
print(f"Chunking comparison (CHUNK_SIZE={CHUNK_SIZE}, topic=True):")
print(pd.DataFrame([
    {"config": f"overlap={CHUNK_OVERLAP}",     **{k: report_default[k]   for k in METRIC_KEYS}},
    {"config": f"overlap={CHUNK_OVERLAP_ALT}", **{k: report_overlap75[k] for k in METRIC_KEYS}},
]).to_string(index=False))

## 5) Latency

We time `recover(query, top_k)` — a single call that covers embedding + FAISS search. Each query runs `N_ITER_LAT` times; the first call is a warm-up and not counted.

In [ ]:
EVAL_QUERIES = [
    "What is RAG?",
    "How does retrieval-augmented generation work?",
    "How to create a git branch?",
    "What is the difference between git merge and git rebase?",
    "What is Google Cloud Platform?",
    "How do I deploy a container on GCP?",
    "What are the advantages of using a vector database?",
    "What is the difference between BM25 and dense retrieval?",
    "What is the capital of France?",           # out-of-scope
    "Who won the FIFA World Cup in 2022?",       # out-of-scope
]

# Maps each query to its expected topic folder; None = out-of-scope
EXPECTED_TOPIC = {
    "What is RAG?": "rag",
    "How does retrieval-augmented generation work?": "rag",
    "How to create a git branch?": "git",
    "What is the difference between git merge and git rebase?": "git",
    "What is Google Cloud Platform?": "gcp",
    "How do I deploy a container on GCP?": "gcp",
    "What are the advantages of using a vector database?": "rag",
    "What is the difference between BM25 and dense retrieval?": "rag",
    "What is the capital of France?": None,
    "Who won the FIFA World Cup in 2022?": None,
}


def measure_latency(query: str, n_iter: int = N_ITER_LAT, top_k: int = TOP_K):
    _ = recover(query, top_k=top_k)  # warm-up — not counted
    rows = []
    for it in range(n_iter):
        t0 = time.perf_counter()
        recover(query, top_k=top_k)
        rows.append({"query": query, "iter": it + 1,
                     "total_ms": round((time.perf_counter() - t0) * 1000, 3)})
    return rows


print(f"Queries: {len(EVAL_QUERIES)}, iterations per query: {N_ITER_LAT}")

In [ ]:
lat_rows = []
for q in EVAL_QUERIES:
    lat_rows.extend(measure_latency(q))

lat_df = pd.DataFrame(lat_rows)
display(
    lat_df.groupby("query")["total_ms"]
    .agg(["min", "mean", "max"])
    .round(3)
)
print(f"Overall mean: {lat_df['total_ms'].mean():.3f} ms")
save_df(lat_df, "week5_latency_runs.csv")

## 6) Retrieval relevance — manual labeling

`recover()` returns the top-k chunks for a query. We assign a relevance label to each result:

- `auto_label` — heuristic: 1 if the chunk's topic folder matches `EXPECTED_TOPIC`, 0 otherwise. Fast but imprecise.
- `manual_label` — human judgment after reading `text_preview`. This is the authoritative label used for Hit@k and Precision@k.

Out-of-scope queries (capital of France, FIFA) get `manual_label=None` and are excluded from official metrics.

In [ ]:
label_rows = []
for q in EVAL_QUERIES:
    expected_kw = EXPECTED_TOPIC.get(q)
    for rank, (chunk, ref, score) in enumerate(recover(q, top_k=TOP_K), 1):
        auto = 0 if expected_kw is None else int(expected_kw in ref["topic"].lower())
        label_rows.append({
            "query":        q,
            "rank":         rank,
            "score":        round(score, 4),
            "topic":        ref["topic"],
            "source":       ref["source"],
            "text_preview": chunk.text[:120].replace("\n", " "),
            "expected_kw":  str(expected_kw),
            "auto_label":   auto,
            "manual_label": None,
        })

label_df = pd.DataFrame(label_rows)
display(label_df[["query", "rank", "score", "topic", "expected_kw", "auto_label", "manual_label", "text_preview"]])
print("Set manual_label (1/0) for at least 5-10 rows, then run the next cell.")
save_df(label_df, "week5_relevance_labels_initial.csv")

In [ ]:
# ── Manual labeling ──────────────────────────────────────────────────────────
# Step 1: seed manual_label from auto_label for all in-scope queries.
#         (out-of-scope rows — 'capital of France', 'FIFA' — stay None)
# Step 2: override specific (query, rank) pairs below where you disagree
#         after reviewing text_preview in the table above.

# Seed from auto_label
in_scope = label_df['expected_kw'] != 'None'
label_df.loc[in_scope, 'manual_label'] = label_df.loc[in_scope, 'auto_label']

# Manual overrides — uncomment and fill where auto_label got it wrong
OVERRIDES = {
    # ('What is RAG?', 1): 1,
    # ('What is RAG?', 2): 0,
    # ('What is the difference between BM25 and dense retrieval?', 1): 0,
    # etc.
}

for (query, rank), label in OVERRIDES.items():
    mask = (label_df['query'] == query) & (label_df['rank'] == rank)
    label_df.loc[mask, 'manual_label'] = label

n_labeled = label_df['manual_label'].notna().sum()
print(f'Labeled: {n_labeled} / {len(label_df)} rows  (out-of-scope rows stay None)')
display(label_df[['query', 'rank', 'score', 'topic', 'expected_kw', 'auto_label', 'manual_label', 'text_preview']])
save_df(label_df, 'week5_relevance_labels_latest.csv')


## 7) Official vs estimated metrics

`compute_label_metrics` computes Hit@k and Precision@k from the labeled table:

- **Official** — uses `manual_label` rows only (authoritative)
- **Estimated** — uses `auto_label` for all rows (heuristic, reported separately)

Run this cell after filling in `manual_label` in the table above.

In [ ]:
# Retrieval metrics from labels

def compute_label_metrics(df: pd.DataFrame, top_k: int = TOP_K):
    tmp = df.copy()
    tmp['manual_label'] = pd.to_numeric(tmp['manual_label'], errors='coerce')

    labeled = tmp[tmp['manual_label'].notna()].copy()
    official_df = None
    if not labeled.empty:
        labeled['manual_label'] = labeled['manual_label'].astype(int)
        official_df = (
            labeled.groupby('query')
            .agg(
                **{f'Hit@{top_k}': ('manual_label', 'max')},
                **{f'Prec@{top_k}': ('manual_label', 'mean')},
                n_labeled=('manual_label', 'count'),
            )
            .reset_index()
        )
        official_df[f'Hit@{top_k}'] = official_df[f'Hit@{top_k}'].astype(int)
        official_df[f'Prec@{top_k}'] = official_df[f'Prec@{top_k}'].round(3)

    estimated_df = (
        tmp.groupby('query')
        .agg(
            **{f'Hit@{top_k}': ('auto_label', 'max')},
            **{f'Prec@{top_k}': ('auto_label', 'mean')},
        )
        .reset_index()
    )
    estimated_df[f'Hit@{top_k}'] = estimated_df[f'Hit@{top_k}'].astype(int)
    estimated_df[f'Prec@{top_k}'] = estimated_df[f'Prec@{top_k}'].round(3)
    estimated_df['out_of_scope'] = estimated_df['query'].map(lambda q: EXPECTED_TOPIC.get(q) is None)

    return official_df, estimated_df


official_df, estimated_df = compute_label_metrics(label_df, top_k=TOP_K)

if official_df is None:
    print('No manual labels yet. Fill manual_label and rerun this cell for official metrics.')
else:
    print('=== OFFICIAL METRICS (manual labels) ===')
    display(official_df)
    print(f"Macro Hit@{TOP_K}: {official_df[f'Hit@{TOP_K}'].mean():.3f}")
    print(f"Macro Prec@{TOP_K}: {official_df[f'Prec@{TOP_K}'].mean():.3f}")

print('\n=== ESTIMATED METRICS (auto labels, heuristic) ===')
display(estimated_df)
print(f"Estimated Macro Hit@{TOP_K}: {estimated_df[f'Hit@{TOP_K}'].mean():.3f}")
print(f"Estimated Macro Prec@{TOP_K}: {estimated_df[f'Prec@{TOP_K}'].mean():.3f}")

if official_df is not None:
    save_df(official_df, 'week5_relevance_official.csv')
save_df(estimated_df, 'week5_relevance_estimated.csv')
save_df(label_df, 'week5_relevance_labels_latest.csv')


## 8) Prompt comparison — baseline vs grounded

We pass the same context (from `recover()`) through two prompt templates to see how the prompt affects answer quality:

- **Baseline** — open-ended; lets the LLM add external knowledge
- **Grounded** — strict citations required; model must respond `Insufficient evidence` when context is weak

This isolates prompt behaviour from retrieval quality.

In [ ]:
from langchain_ollama import OllamaLLM

BASE_PROMPT = """You are a RAG assistant.
Use ONLY the context below. If there is not enough information, say: I don't know based on provided context.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

OPT_PROMPT = """You are a strict grounded assistant.
Rules:
1) Answer using ONLY the CONTEXT chunks below.
2) If evidence is missing or weak, answer exactly: Insufficient evidence in retrieved context.
3) Cite every factual claim with [Chunk N] where N is the chunk number.
4) Do not add external facts or prior knowledge.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""


def build_context_str(query: str, top_k: int = TOP_K) -> tuple:
    hits  = recover(query, top_k=top_k)
    lines = [
        f"[Chunk {rank}] (score={score:.4f}, source={ref['source']}, topic={ref['topic']})\n{chunk.text[:500]}"
        for rank, (chunk, ref, score) in enumerate(hits, 1)
    ]
    return "\n\n".join(lines), [ref for _, ref, _ in hits]


def try_llm(prompt_text: str) -> tuple:
    try:
        llm = OllamaLLM(model="gemma3:4b", base_url="http://localhost:11434",
                        temperature=0.0, validate_model_on_init=True)
        return (llm.invoke(prompt_text) or "").strip(), None
    except Exception as e:
        return "", str(e)


PROMPT_QUERIES = [
    "What is RAG?",
    "How to create a git branch?",
    "What is Google Cloud Platform?",
]

prompt_rows = []
for q in PROMPT_QUERIES:
    context, refs = build_context_str(q)
    a_base, e_base = try_llm(BASE_PROMPT.format(context=context, query=q))
    a_opt,  e_opt  = try_llm(OPT_PROMPT.format(context=context, query=q))
    topics = ",".join(r["topic"] for r in refs)
    prompt_rows += [
        {"query": q, "variant": "baseline", "error": e_base, "answer": a_base[:700], "topics": topics},
        {"query": q, "variant": "grounded", "error": e_opt,  "answer": a_opt[:700],  "topics": topics},
    ]

prompt_df = pd.DataFrame(prompt_rows)
display(prompt_df[["query", "variant", "error", "answer"]])
save_df(prompt_df, "week5_prompt_comparison.csv")

## 9) Conclusions

| What we measured | How | What to do if it's bad |
|------------------|-----|------------------------|
| Retrieval quality | hit@k and MRR@5 via `evaluate()` | Tune `CHUNK_SIZE`, `CHUNK_OVERLAP`, or `TOP_K` |
| Topic filtering | Compare `use_topic=True/False` | Disable if topic labels are noisy; enable if cross-topic chunks pollute results |
| Chunking overlap | Rebuild index with different `chunk_overlap` | Increase overlap if boundary cuts hurt recall |
| Latency | `recover()` timed over 20 iterations | Reduce embedding dim or pre-embed the corpus |
| Prompt faithfulness | Baseline vs grounded prompt on same context | Use the grounded prompt in production; baseline leaks parametric knowledge |